# Stage 3C — Exploratory Algorithm Benchmark and Diagnostic Audit

**Purpose:** compare prespecified simple, regularized, kernel, bagged-tree, and boosted-tree classifiers without changing or overwriting the completed Stage 3B primary analysis.

## Scientific status

- Stage 3B common-core ridge remains the **prespecified confirmatory primary model**.
- Every Stage 3C model is explicitly **post hoc exploratory**.
- No SMOTE, synthetic observations, undersampling, class weighting, or outcome-dependent case deletion is used.
- All candidate algorithms use the same leakage-controlled broad predictor set and the exact saved Stage 3B outer splits.
- Preprocessing and hyperparameter tuning occur only inside training data.
- Primary tuning criterion remains log loss.
- Model comparison uses averaged patient-level out-of-fold probabilities and paired bootstrap differences.
- No clinical threshold, net-benefit claim, or clinical deployment claim is made.

## Prespecified benchmark

1. Prevalence-only null model
2. Refined broad ridge logistic regression
3. Broad lasso logistic regression
4. Refined broad elastic-net logistic regression
5. Linear support-vector classifier
6. Radial-basis-function support-vector classifier
7. Random forest
8. XGBoost
9. AdaBoost

The notebook creates auditable tables, diagnostic figures, signed per-outcome/per-model checkpoints, an artifact registry, and a final self-contained HTML report.

In [ ]:
# 1. Imports, versions, and reproducible runtime settings
from __future__ import annotations
import hashlib, json, math, os, pickle, platform, subprocess, sys, time, warnings
from dataclasses import dataclass
from pathlib import Path
from packaging.version import Version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, Markdown, Image
from scipy.special import expit, logit

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError as e:
    raise ImportError('Stage 3C requires xgboost. Install with: python -m pip install "xgboost>=2.1,<4"') from e

SKLEARN_VERSION=Version(sklearn.__version__)
if SKLEARN_VERSION < Version('1.8'):
    raise RuntimeError(f'scikit-learn >=1.8,<1.10 is required; found {sklearn.__version__}')

MASTER_SEED=20261017
np.random.seed(MASTER_SEED)
pd.set_option('display.max_columns',180); pd.set_option('display.max_rows',180)
pd.set_option('display.width',220); pd.set_option('display.float_format',lambda x:f'{x:,.5f}')
sns.set_theme(style='whitegrid',context='notebook',palette='colorblind')
print('Python:',sys.version.split()[0]); print('Platform:',platform.platform())
print('pandas:',pd.__version__); print('scikit-learn:',sklearn.__version__); print('xgboost:',xgb.__version__)
print('Deprecated LogisticRegression penalty argument is not used.')

In [ ]:
# 2. Portable project discovery and strictly separate Stage 3C outputs
NOTEBOOK_NAME='03C_exploratory_algorithm_benchmark.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()
COLAB_PROJECT=Path.cwd()
ENV_PROJECT=Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else None
candidate_roots=[p for p in [ENV_PROJECT,ENV_PROJECT,ENV_PROJECT/'stage2_outputs',COLAB_PROJECT,
                              Path.cwd(),Path.cwd()/'stage2_outputs',Path.cwd()/'ra_analysis'/'stage2_outputs',
                              Path.cwd()/'stage2_outputs'] if p is not None]
PROJECT_DIR=next((p for p in candidate_roots if (p/'cleaned_data'/'cleaned_master_data.csv').exists()),None)
if PROJECT_DIR is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR=next((p for p in candidate_roots if (p/'cleaned_data'/'cleaned_master_data.csv').exists()),None)
    except (ImportError,ModuleNotFoundError):
        pass
if PROJECT_DIR is None:
    raise FileNotFoundError('Project not found. Set RA_PROJECT_DIR to the authorized local project directory.')

protocol_candidates=[ENV_PROJECT/'config'/'stage3_modeling_decisions.yaml',PROJECT_DIR/'config'/'stage3_modeling_decisions.yaml',
                     PROJECT_DIR/'stage3_modeling_decisions.yaml',PROJECT_DIR.parent/'stage3_modeling_decisions.yaml',
                     Path.cwd()/'stage3_modeling_decisions.yaml',ENV_PROJECT/'stage3_modeling_decisions.yaml']
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
if PROTOCOL_PATH is None: raise FileNotFoundError('stage3_modeling_decisions.yaml not found')

DERIVED_DIR=PROJECT_DIR/'derived_data'
CLEAN_PATH=PROJECT_DIR/'cleaned_data'/'cleaned_master_data.csv'
STAGE2_AUDIT=PROJECT_DIR/'preprocessing_outputs'/'tables'/'stage2_validation_audit.csv'
STAGE3A_AUDIT=PROJECT_DIR/'modeling_feasibility'/'tables'/'stage3a_validation_audit.csv'
STAGE3B_DIR=PROJECT_DIR/'modeling_results'
STAGE3B_TABLES=STAGE3B_DIR/'tables'
STAGE3B_SPLITS=STAGE3B_TABLES/'saved_outer_split_assignments.csv'
STAGE3B_PREDS=STAGE3B_TABLES/'patient_averaged_oof_predictions.csv'
STAGE3B_AUDIT=STAGE3B_TABLES/'stage3b_validation_audit.csv'

# Never write Stage 3C artifacts into Stage 3B.
OUT_DIR=PROJECT_DIR/'modeling_results_stage3c'
TABLE_DIR=OUT_DIR/'tables'; FIGURE_DIR=OUT_DIR/'figures'; CHECKPOINT_DIR=OUT_DIR/'checkpoints'; REPORT_DIR=OUT_DIR/'report'
for d in [OUT_DIR,TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,REPORT_DIR]: d.mkdir(parents=True,exist_ok=True)

QUICK_MODE=os.environ.get('RA_STAGE3C_QUICK_TEST','0')=='1'
FORCE_RECOMPUTE=os.environ.get('RA_STAGE3C_FORCE_RECOMPUTE','0')=='1'
XGB_DEVICE=os.environ.get('RA_STAGE3C_XGB_DEVICE','cpu')
if QUICK_MODE:
    EXPECTED_REPEATS,EXPECTED_FOLDS,INNER_SPLITS,BOOTSTRAPS,PERM_REPEATS=1,3,2,50,1
else:
    EXPECTED_REPEATS,EXPECTED_FOLDS,INNER_SPLITS,BOOTSTRAPS,PERM_REPEATS=10,5,5,2000,1
N_JOBS=-1
OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
CORE_REFERENCE_MODELS=['core_ridge','broad_ridge','broad_elastic_net']
print('Project:',PROJECT_DIR.resolve()); print('Stage 3C output:',OUT_DIR.resolve())
print('Mode:','QUICK SOFTWARE TEST' if QUICK_MODE else 'FULL EXPLORATORY SCIENTIFIC BENCHMARK')
print('Expected outer design:',EXPECTED_FOLDS,'folds x',EXPECTED_REPEATS,'repeats | inner:',INNER_SPLITS)
print('CPU cores:',os.cpu_count(),'| search n_jobs:',N_JOBS,'| XGBoost device:',XGB_DEVICE)

## Prerequisite and immutability checks

In [ ]:
# 3. Accept only verified prior stages; fingerprint Stage 3B before analysis
def sha256_file(path:Path)->str:
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

for p,label in [(STAGE2_AUDIT,'Stage 2'),(STAGE3A_AUDIT,'Stage 3A'),(STAGE3B_AUDIT,'Stage 3B')]:
    if not p.exists(): raise FileNotFoundError(f'{label} audit missing: {p}')
    a=pd.read_csv(p)
    if a.empty or not a.status.eq('PASS').all(): raise AssertionError(f'{label} audit did not fully pass')
if not QUICK_MODE:
    b=pd.read_csv(STAGE3B_AUDIT)
    detail=' '.join(b.details.fillna('').astype(str))
    if 'FULL scientific mode' not in detail: raise AssertionError('Full Stage 3C requires full Stage 3B artifacts')

for p in [CLEAN_PATH,STAGE3B_SPLITS,STAGE3B_PREDS,PROTOCOL_PATH]:
    if not p.exists(): raise FileNotFoundError(p)
stage3b_immutable_paths=[p for p in [STAGE3B_SPLITS,STAGE3B_PREDS,STAGE3B_AUDIT] if p.exists()]
STAGE3B_HASHES_BEFORE={str(p):sha256_file(p) for p in stage3b_immutable_paths}
clean=pd.read_csv(CLEAN_PATH); assert len(clean)==820 and clean.VisitID.is_unique
with open(PROTOCOL_PATH,encoding='utf-8') as f: protocol=yaml.safe_load(f)
assert protocol['model_hierarchy']['confirmatory_primary']['model']=='ridge logistic regression'
print('Prior stages accepted; Stage 3B fingerprints recorded and will be rechecked.')

In [ ]:
# 4. Validate and index the exact saved Stage 3B outer splits
saved_splits=pd.read_csv(STAGE3B_SPLITS)
required={'VisitID','repeat','fold','outcome'}
if not required.issubset(saved_splits.columns): raise AssertionError(required-set(saved_splits.columns))
assert set(saved_splits.outcome)==set(OUTCOMES)
for outcome,g in saved_splits.groupby('outcome'):
    if g['repeat'].nunique()!=EXPECTED_REPEATS: raise AssertionError((outcome,'repeat count',g['repeat'].nunique()))
    for repeat,rg in g.groupby('repeat'):
        if rg.fold.nunique()!=EXPECTED_FOLDS: raise AssertionError((outcome,repeat,'fold count'))
        if rg.VisitID.nunique()!=820 or len(rg)!=820: raise AssertionError((outcome,repeat,'patient coverage'))
print('Saved Stage 3B split assignments validated:',saved_splits.shape)

In [ ]:
# 5. Signed Stage 3C run configuration
if QUICK_MODE:
    search_budget={'enet':4,'rf':4,'xgb':4,'ada':4}
else:
    search_budget={'enet':40,'rf':30,'xgb':50,'ada':24}
signature_payload={
    'pipeline_version':'3C-1.2-calibrated-svc-compatible',
    'clean_sha256':sha256_file(CLEAN_PATH),'protocol_sha256':sha256_file(PROTOCOL_PATH),
    'stage3b_splits_sha256':sha256_file(STAGE3B_SPLITS),'stage3b_predictions_sha256':sha256_file(STAGE3B_PREDS),
    'sklearn_version':sklearn.__version__,'xgboost_version':xgb.__version__,
    'quick_mode':QUICK_MODE,'outer_repeats':EXPECTED_REPEATS,'outer_folds':EXPECTED_FOLDS,
    'inner_splits':INNER_SPLITS,'bootstraps':BOOTSTRAPS,'seed':MASTER_SEED,
    'xgb_device':XGB_DEVICE,'search_budget':search_budget,
    'sampling':'natural prevalence; no SMOTE, oversampling, undersampling, or class weights',
}
RUN_SIGNATURE=hashlib.sha256(json.dumps(signature_payload,sort_keys=True).encode()).hexdigest()
with open(OUT_DIR/'run_signature.json','w',encoding='utf-8') as f:
    json.dump({'run_signature':RUN_SIGNATURE,**signature_payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)

## Leakage-controlled preprocessing and model definitions

In [ ]:
# 6. Same leakage-controlled broad predictor construction used in Stage 3B
class MissingnessFlags(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): self.n_features_in_=X.shape[1]; return self
    def transform(self,X):
        return pd.isna(X).astype(float).to_numpy() if hasattr(X,'to_numpy') else np.isnan(X).astype(float)
    def get_feature_names_out(self,input_features=None):
        if input_features is None: input_features=[f'x{i}' for i in range(self.n_features_in_)]
        return np.asarray([f'{x}__missing' for x in input_features],dtype=object)

CATEGORY_LEVELS={'Gender':['Female','Male'],'RAOnset':['Insidious','Acute','Unknown'],
                 'PhysicalActivity':['Moderate','Low','High']}

def prepare_broad_data(outcome):
    data=pd.read_csv(DERIVED_DIR/f'{outcome}__primary_eligible.csv')
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    predictors=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
    if outcome in predictors or 'VisitID' in predictors: raise AssertionError('Outcome or ID leakage')
    data['RADurationDays_log1p']=np.log1p(data['RADurationDays'])
    predictors=['RADurationDays_log1p' if x=='RADurationDays' else x for x in predictors]
    semantic=dict(zip(manifest.column,manifest.semantic_type)); semantic['RADurationDays_log1p']='numeric'; semantic.pop('RADurationDays',None)
    if len(data)!=820 or not data.VisitID.is_unique: raise AssertionError('Unexpected analysis population')
    return data,predictors,semantic

def make_preprocessor(data,predictors,semantic):
    numeric=[x for x in predictors if semantic[x]=='numeric']
    binary=[x for x in predictors if semantic[x]=='binary']
    categorical=[x for x in predictors if semantic[x]=='categorical']
    indicators=[x for x in numeric+binary if data[x].isna().any()]
    tr=[]
    if numeric: tr.append(('numeric',Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),
                                               ('scaler',StandardScaler())]),numeric))
    if binary: tr.append(('binary',Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True))]),binary))
    if categorical:
        categories=[]
        for col in categorical:
            if col not in CATEGORY_LEVELS: raise KeyError(f'No frozen levels for {col}')
            observed=set(data[col].dropna().astype(str).unique())
            allowed=set(CATEGORY_LEVELS[col])-{'Unknown'}
            if not observed.issubset(allowed): raise ValueError((col,observed-allowed))
            categories.append(CATEGORY_LEVELS[col])
        tr.append(('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Unknown',keep_empty_features=True)),
                                           ('onehot',OneHotEncoder(categories=categories,drop='first',handle_unknown='ignore',sparse_output=True))]),categorical))
    if indicators: tr.append(('missing',MissingnessFlags(),indicators))
    return ColumnTransformer(tr,remainder='drop',verbose_feature_names_out=True)

In [ ]:
# 7. Prespecified candidate models and bounded tuning spaces
@dataclass
class ModelSpec:
    name:str
    estimator:object
    params:dict
    search:str='grid'
    n_iter:int|None=None

QUICK=QUICK_MODE
ridge_C=[0.01,0.1] if QUICK else np.logspace(np.log10(0.003),np.log10(0.5),21).tolist()
lasso_C=[0.01,0.1] if QUICK else np.logspace(-4,1,16).tolist()
enet_C=[0.01,0.1] if QUICK else np.logspace(np.log10(0.003),np.log10(0.5),16).tolist()
enet_l1=[0.1,0.9] if QUICK else [0.0,0.05,0.10,0.25,0.50,0.75,0.90,1.0]
svc_C=[0.1,1.0] if QUICK else [0.01,0.1,1,10,100]
svc_gamma=['scale',0.01] if QUICK else ['scale',0.001,0.01,0.1]
CALIBRATION_CV=2 if QUICK else 3
rf_params={'model__max_depth':[3,None] if QUICK else [3,5,8,None],
           'model__min_samples_leaf':[5,20] if QUICK else [5,10,20],
           'model__max_features':['sqrt'] if QUICK else ['sqrt',0.3,0.6],
           'model__n_estimators':[150] if QUICK else [500,800]}
xgb_params={'model__max_depth':[1,2] if QUICK else [1,2,3],
            'model__learning_rate':[0.05,0.1] if QUICK else [0.01,0.03,0.05,0.1],
            'model__min_child_weight':[5] if QUICK else [5,10,20],
            'model__subsample':[0.8] if QUICK else [0.7,0.85,1.0],
            'model__colsample_bytree':[0.75] if QUICK else [0.5,0.75,1.0],
            'model__reg_lambda':[10] if QUICK else [1,10,50],
            'model__reg_alpha':[0] if QUICK else [0,0.1,1],
            'model__n_estimators':[150] if QUICK else [300,600,1000]}
ada_params={'model__n_estimators':[50,100] if QUICK else [50,100,200,400],
            'model__learning_rate':[0.05,0.1] if QUICK else [0.01,0.05,0.1,0.5],
            'model__estimator__max_depth':[1] if QUICK else [1,2]}

MODEL_SPECS=[
    ModelSpec('refined_broad_ridge',LogisticRegression(l1_ratio=0.0,solver='lbfgs',max_iter=10000,random_state=MASTER_SEED),{'model__C':ridge_C}),
    ModelSpec('broad_lasso',LogisticRegression(l1_ratio=1.0,solver='saga',max_iter=20000,tol=1e-4,random_state=MASTER_SEED),{'model__C':lasso_C}),
    ModelSpec('refined_broad_elastic_net',LogisticRegression(l1_ratio=0.5,solver='saga',max_iter=20000,tol=1e-4,random_state=MASTER_SEED),
              {'model__C':enet_C,'model__l1_ratio':enet_l1},'random',search_budget['enet']),
    ModelSpec('linear_svc',CalibratedClassifierCV(
                  estimator=SVC(kernel='linear',random_state=MASTER_SEED,cache_size=1500),
                  method='sigmoid',cv=CALIBRATION_CV,ensemble=False),
              {'model__estimator__C':svc_C}),
    ModelSpec('rbf_svc',CalibratedClassifierCV(
                  estimator=SVC(kernel='rbf',random_state=MASTER_SEED,cache_size=1500),
                  method='sigmoid',cv=CALIBRATION_CV,ensemble=False),
              {'model__estimator__C':svc_C,'model__estimator__gamma':svc_gamma}),
    ModelSpec('random_forest',RandomForestClassifier(random_state=MASTER_SEED,n_jobs=1,class_weight=None),rf_params,'random',search_budget['rf']),
    ModelSpec('xgboost',XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',device=XGB_DEVICE,
                                      random_state=MASTER_SEED,n_jobs=1,scale_pos_weight=1),xgb_params,'random',search_budget['xgb']),
    ModelSpec('adaboost',AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,random_state=MASTER_SEED),random_state=MASTER_SEED),
              ada_params,'random',search_budget['ada']),
]
MODEL_ORDER=['prevalence_null']+[s.name for s in MODEL_SPECS]
model_inventory=[]
for s in MODEL_SPECS:
    total=int(np.prod([len(v) for v in s.params.values()]))
    model_inventory.append({'model':s.name,'family':type(s.estimator).__name__,'search':s.search,
                            'available_combinations':total,'evaluated_candidates':total if s.search=='grid' else min(total,s.n_iter)})
model_inventory=pd.DataFrame(model_inventory)
model_inventory.to_csv(TABLE_DIR/'model_inventory_and_search_budget.csv',index=False,encoding='utf-8-sig')
display(model_inventory)

## Metrics, bootstrap inference, and diagnostics

In [ ]:
# 8. Performance functions
EPS=1e-8
def calibration_stats(y,p):
    p=np.clip(np.asarray(p,float),EPS,1-EPS); y=np.asarray(y,int)
    if np.std(logit(p))<1e-10: return np.nan,np.nan
    try:
        slope_model=LogisticRegression(C=np.inf,l1_ratio=0.0,solver='lbfgs',max_iter=5000).fit(logit(p).reshape(-1,1),y)
        slope=float(slope_model.coef_[0,0]); intercept=float(slope_model.intercept_[0])
        return intercept,slope
    except Exception: return np.nan,np.nan

def metric_dict(y,p):
    ci,cs=calibration_stats(y,p)
    return {'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),
            'log_loss':log_loss(y,p,labels=[0,1]),'brier':brier_score_loss(y,p),
            'calibration_intercept':ci,'calibration_slope':cs}

def stratified_bootstrap_metrics(frame,n_boot,seed):
    rng=np.random.default_rng(seed); y=frame.y.to_numpy(int); p=frame.probability.to_numpy(float)
    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0); rows=[]
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)])
        rows.append(metric_dict(y[idx],p[idx]))
    return pd.DataFrame(rows)

def threshold_metrics(y,p,t):
    pred=np.asarray(p)>=t; tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    safe=lambda a,b: a/b if b else np.nan
    return {'threshold':t,'tn':tn,'fp':fp,'fn':fn,'tp':tp,'sensitivity':safe(tp,tp+fn),
            'specificity':safe(tn,tn+fp),'ppv':safe(tp,tp+fp),'npv':safe(tn,tn+fn)}

In [ ]:
# 9. Helpers for exact saved splits, search, convergence, and checkpoints
def split_indices_for(data,outcome):
    id_to_idx=pd.Series(np.arange(len(data)),index=data.VisitID).to_dict()
    rows=[]
    sg=saved_splits[saved_splits.outcome.eq(outcome)]
    all_ids=set(data.VisitID)
    for repeat in sorted(sg.repeat.unique()):
        rg=sg[sg.repeat.eq(repeat)]
        for fold in sorted(rg.fold.unique()):
            test_ids=rg.loc[rg.fold.eq(fold),'VisitID'].tolist()
            test_idx=np.array([id_to_idx[x] for x in test_ids],int)
            train_idx=np.array([id_to_idx[x] for x in data.VisitID if x not in set(test_ids)],int)
            if set(data.VisitID.iloc[train_idx]) & set(test_ids): raise AssertionError('Split overlap')
            if set(data.VisitID.iloc[train_idx]) | set(test_ids) != all_ids: raise AssertionError('Split omission')
            rows.append((int(repeat),int(fold),train_idx,test_idx))
    return rows

def make_search(pipe,spec,inner,seed):
    common=dict(estimator=pipe,scoring='neg_log_loss',cv=inner,refit=True,n_jobs=N_JOBS,
                pre_dispatch='2*n_jobs',error_score='raise',return_train_score=True)
    if spec.search=='grid': return GridSearchCV(param_grid=spec.params,**common)
    total=int(np.prod([len(v) for v in spec.params.values()]))
    return RandomizedSearchCV(param_distributions=spec.params,n_iter=min(total,spec.n_iter),random_state=seed,**common)

def convergence_diagnostic(fitted):
    model=fitted.named_steps['model']; out={'fit_status':getattr(model,'fit_status_',0),'n_iter_max':np.nan,
                                           'max_iter':getattr(model,'max_iter',np.nan),'convergence_ok':True}
    if hasattr(model,'n_iter_'):
        n=np.asarray(model.n_iter_,dtype=float); out['n_iter_max']=float(np.nanmax(n))
        if np.isfinite(out['max_iter']) and out['max_iter']>0: out['convergence_ok']=bool(out['n_iter_max']<out['max_iter'])
    if getattr(model,'fit_status_',0)!=0: out['convergence_ok']=False
    return out

def checkpoint_path(outcome,model):
    return CHECKPOINT_DIR/f'{outcome}__{model}__{"quick" if QUICK_MODE else "full"}.pkl'

## Repeated nested model fitting

In [ ]:
# 10. Fit one candidate model for one outcome; save an independent signed checkpoint
def fit_candidate(outcome,spec):
    cp=checkpoint_path(outcome,spec.name)
    if cp.exists() and not FORCE_RECOMPUTE:
        with open(cp,'rb') as f: saved=pickle.load(f)
        if saved.get('run_signature')==RUN_SIGNATURE:
            print('Loading:',cp.name); return saved['result']
        print('Ignoring mismatched checkpoint:',cp.name)
    data,predictors,semantic=prepare_broad_data(outcome)
    X=data[predictors]; y=data[outcome].astype(int).to_numpy(); ids=data.VisitID.to_numpy()
    split_plan=split_indices_for(data,outcome)
    predictions=[]; fold_metrics=[]; hyper=[]; diagnostics=[]; importance=[]; runtime=[]
    start_all=time.time()
    for split_no,(repeat,fold,train_idx,test_idx) in enumerate(split_plan):
        inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=MASTER_SEED+1000*repeat+fold)
        prep=make_preprocessor(data.iloc[train_idx],predictors,semantic)
        pipe=Pipeline([('preprocess',prep),('model',clone(spec.estimator))],memory=None)
        search=make_search(pipe,spec,inner,MASTER_SEED+1000*repeat+fold)
        t0=time.time()
        with threadpool_limits(limits=1,user_api='blas'):
            search.fit(X.iloc[train_idx],y[train_idx])
        fitted=search.best_estimator_; elapsed=time.time()-t0
        p=fitted.predict_proba(X.iloc[test_idx])[:,1]
        for j,idx in enumerate(test_idx):
            predictions.append({'VisitID':ids[idx],'row_index':idx,'outcome':outcome,'model':spec.name,
                                'repeat':repeat,'fold':fold,'y':int(y[idx]),'probability':float(p[j])})
        fold_metrics.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                             'n_test':len(test_idx),'events_test':int(y[test_idx].sum()),**metric_dict(y[test_idx],p)})
        bi=search.best_index_; best_params=search.best_params_
        hyper.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                      'best_params_json':json.dumps(best_params,sort_keys=True,default=float),
                      'best_inner_log_loss':-float(search.cv_results_['mean_test_score'][bi]),
                      'best_inner_train_log_loss':-float(search.cv_results_['mean_train_score'][bi]),
                      'inner_generalization_gap':float(search.cv_results_['mean_test_score'][bi]-search.cv_results_['mean_train_score'][bi]),
                      'candidate_count':len(search.cv_results_['params'])})
        diagnostics.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,**convergence_diagnostic(fitted)})
        runtime.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,'fit_seconds':elapsed})
        # Common model-agnostic, outer-test permutation importance at source-predictor level.
        pi=permutation_importance(fitted,X.iloc[test_idx],y[test_idx],scoring='neg_log_loss',
                                  n_repeats=PERM_REPEATS,random_state=MASTER_SEED+1000*repeat+fold,n_jobs=1)
        for feature,mean,sd in zip(predictors,pi.importances_mean,pi.importances_std):
            importance.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                               'feature':'RADurationDays' if feature=='RADurationDays_log1p' else feature,
                               'permutation_logloss_importance':float(mean),'permutation_sd':float(sd)})
        if (split_no+1)%EXPECTED_FOLDS==0:
            print(f'{outcome} | {spec.name}: repeat {repeat+1}/{EXPECTED_REPEATS} | {(time.time()-start_all)/60:.1f} min')
    result={'predictions':pd.DataFrame(predictions),'fold_metrics':pd.DataFrame(fold_metrics),
            'hyperparameters':pd.DataFrame(hyper),'diagnostics':pd.DataFrame(diagnostics),
            'importance':pd.DataFrame(importance),'runtime':pd.DataFrame(runtime)}
    with open(cp,'wb') as f: pickle.dump({'run_signature':RUN_SIGNATURE,'result':result},f,pickle.HIGHEST_PROTOCOL)
    print('Saved:',cp.name); return result

In [ ]:
# 11. Fit/load all candidates and outcomes
all_results={}
for outcome in OUTCOMES:
    for spec in MODEL_SPECS:
        all_results[(outcome,spec.name)]=fit_candidate(outcome,spec)
print('All Stage 3C candidate pipelines completed.')

In [ ]:
# 12. Add prevalence-only OOF predictions on the exact same outer splits
null_rows=[]; null_fold=[]
for outcome in OUTCOMES:
    data,_,_=prepare_broad_data(outcome); y=data[outcome].astype(int).to_numpy(); ids=data.VisitID.to_numpy()
    for repeat,fold,train_idx,test_idx in split_indices_for(data,outcome):
        p=np.repeat(y[train_idx].mean(),len(test_idx))
        null_fold.append({'outcome':outcome,'model':'prevalence_null','repeat':repeat,'fold':fold,
                          'n_test':len(test_idx),'events_test':int(y[test_idx].sum()),**metric_dict(y[test_idx],p)})
        for j,idx in enumerate(test_idx):
            null_rows.append({'VisitID':ids[idx],'row_index':idx,'outcome':outcome,'model':'prevalence_null',
                              'repeat':repeat,'fold':fold,'y':int(y[idx]),'probability':float(p[j])})
null_predictions=pd.DataFrame(null_rows); null_fold_metrics=pd.DataFrame(null_fold)

## Aggregation, uncertainty, and paired comparisons

In [ ]:
# 13. Aggregate every auditable result table
predictions=pd.concat([null_predictions]+[r['predictions'] for r in all_results.values()],ignore_index=True)
fold_metrics=pd.concat([null_fold_metrics]+[r['fold_metrics'] for r in all_results.values()],ignore_index=True)
hyperparameters=pd.concat([r['hyperparameters'] for r in all_results.values()],ignore_index=True)
diagnostics=pd.concat([r['diagnostics'] for r in all_results.values()],ignore_index=True)
importance=pd.concat([r['importance'] for r in all_results.values()],ignore_index=True)
runtime=pd.concat([r['runtime'] for r in all_results.values()],ignore_index=True)

avg_predictions=(predictions.groupby(['outcome','model','VisitID','y'],as_index=False)
                 .agg(probability=('probability','mean'),oof_repetitions=('probability','size'),
                      probability_sd_across_repeats=('probability','std')))
for frame,name in [(predictions,'all_repeated_oof_predictions.csv'),(avg_predictions,'patient_averaged_oof_predictions.csv'),
                   (fold_metrics,'outer_fold_metric_stability.csv'),(hyperparameters,'selected_hyperparameters.csv'),
                   (diagnostics,'convergence_diagnostics.csv'),(importance,'outer_test_permutation_importance.csv'),
                   (runtime,'fit_runtime.csv')]:
    frame.to_csv(TABLE_DIR/name,index=False,encoding='utf-8-sig')
print('Repeated predictions:',predictions.shape,'| averaged:',avg_predictions.shape)

In [ ]:
# 14. Point estimates and 2,000 patient-level bootstrap confidence intervals
point_rows=[]; ci_rows=[]
for (outcome,model),g in avg_predictions.groupby(['outcome','model'],sort=False):
    m=metric_dict(g.y,g.probability)
    point_rows.append({'outcome':outcome,'model':model,'n':len(g),'events':int(g.y.sum()),**m})
    boot=stratified_bootstrap_metrics(g,BOOTSTRAPS,MASTER_SEED+OUTCOMES.index(outcome)*100+MODEL_ORDER.index(model))
    for metric,value in m.items():
        vals=boot[metric].dropna()
        ci_rows.append({'outcome':outcome,'model':model,'metric':metric,'estimate':value,
                        'ci_low':vals.quantile(.025) if len(vals) else np.nan,
                        'ci_high':vals.quantile(.975) if len(vals) else np.nan,
                        'bootstrap_resamples':BOOTSTRAPS,'valid_resamples':len(vals)})
point_performance=pd.DataFrame(point_rows); performance_ci=pd.DataFrame(ci_rows)
point_performance.to_csv(TABLE_DIR/'point_estimate_performance.csv',index=False,encoding='utf-8-sig')
performance_ci.to_csv(TABLE_DIR/'performance_with_bootstrap_ci.csv',index=False,encoding='utf-8-sig')
display(point_performance)

In [ ]:
# 15. Paired bootstrap comparisons against refined ridge and Stage 3B reference predictions
stage3b_ref=pd.read_csv(STAGE3B_PREDS)
stage3b_ref=stage3b_ref[stage3b_ref.model.isin(CORE_REFERENCE_MODELS)].copy()
stage3b_ref['model']='stage3b_'+stage3b_ref.model
stage3b_ref[['outcome','model','VisitID','y','probability']].to_csv(TABLE_DIR/'stage3b_reference_predictions_read_only.csv',index=False,encoding='utf-8-sig')
combined_for_comparison=pd.concat([avg_predictions[['outcome','model','VisitID','y','probability']],
                                   stage3b_ref[['outcome','model','VisitID','y','probability']]],ignore_index=True)

def paired_difference(outcome,comparator,reference,n_boot,seed):
    a=combined_for_comparison.query('outcome==@outcome and model==@comparator')
    b=combined_for_comparison.query('outcome==@outcome and model==@reference')
    m=a.merge(b,on=['outcome','VisitID','y'],suffixes=('_a','_b'),validate='one_to_one')
    if len(m)!=820: raise AssertionError((outcome,comparator,reference,len(m)))
    y=m.y.to_numpy(int); pa=m.probability_a.to_numpy(); pb=m.probability_b.to_numpy()
    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0); rng=np.random.default_rng(seed)
    def diffs(idx):
        ma=metric_dict(y[idx],pa[idx]); mb=metric_dict(y[idx],pb[idx])
        return {'auroc':ma['auroc']-mb['auroc'],'average_precision':ma['average_precision']-mb['average_precision'],
                'log_loss':mb['log_loss']-ma['log_loss'],'brier':mb['brier']-ma['brier']}
    est=diffs(np.arange(len(y))); rows=[]
    boots={k:[] for k in est}
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)])
        d=diffs(idx)
        for k,v in d.items(): boots[k].append(v)
    for k,v in est.items():
        rows.append({'outcome':outcome,'comparator':comparator,'reference':reference,'metric':k,
                     'improvement_over_reference':v,'ci_low':np.quantile(boots[k],.025),'ci_high':np.quantile(boots[k],.975),
                     'positive_value_favors':'comparator'})
    return rows

comparison_rows=[]
references=['refined_broad_ridge','stage3b_core_ridge','stage3b_broad_ridge']
for outcome in OUTCOMES:
    for comparator in [m for m in MODEL_ORDER if m!='prevalence_null']:
        for reference in references:
            if comparator==reference: continue
            comparison_rows.extend(paired_difference(outcome,comparator,reference,BOOTSTRAPS,
                                                       MASTER_SEED+OUTCOMES.index(outcome)*1000+MODEL_ORDER.index(comparator)*10+references.index(reference)))
paired_comparisons=pd.DataFrame(comparison_rows)
paired_comparisons.to_csv(TABLE_DIR/'paired_model_differences.csv',index=False,encoding='utf-8-sig')
display(paired_comparisons.query("reference=='stage3b_broad_ridge' and metric in ['auroc','log_loss']"))

In [ ]:
# 16. Fold stability, tuning stability, runtime, importance stability, and threshold grids
fold_summary=(fold_metrics.groupby(['outcome','model'])[['auroc','average_precision','log_loss','brier']]
              .agg(['median','std','min','max']).reset_index())
fold_summary.columns=['__'.join([str(x) for x in c if str(x)]) for c in fold_summary.columns]
fold_summary.to_csv(TABLE_DIR/'outer_fold_metric_stability_summary.csv',index=False,encoding='utf-8-sig')

runtime_summary=(runtime.groupby(['outcome','model'],as_index=False)
                 .agg(total_fit_minutes=('fit_seconds',lambda x:x.sum()/60),median_fold_seconds=('fit_seconds','median'),
                      max_fold_seconds=('fit_seconds','max')))
runtime_summary.to_csv(TABLE_DIR/'runtime_summary.csv',index=False,encoding='utf-8-sig')

importance_summary=(importance.groupby(['outcome','model','feature'],as_index=False)
                    .agg(mean_importance=('permutation_logloss_importance','mean'),
                         median_importance=('permutation_logloss_importance','median'),
                         importance_sd=('permutation_logloss_importance','std'),
                         positive_fraction=('permutation_logloss_importance',lambda x:np.mean(x>0)),
                         n_outer_folds=('permutation_logloss_importance','size')))
importance_summary['rank']=importance_summary.groupby(['outcome','model']).mean_importance.rank(ascending=False,method='min')
importance_summary.to_csv(TABLE_DIR/'permutation_importance_summary.csv',index=False,encoding='utf-8-sig')

threshold_rows=[]
for (outcome,model),g in avg_predictions.groupby(['outcome','model']):
    for t in np.arange(.05,.51,.05): threshold_rows.append({'outcome':outcome,'model':model,**threshold_metrics(g.y,g.probability,float(t))})
threshold_table=pd.DataFrame(threshold_rows)
threshold_table.to_csv(TABLE_DIR/'descriptive_threshold_grid_no_selection.csv',index=False,encoding='utf-8-sig')

display(fold_summary); display(runtime_summary); display(importance_summary.query('rank<=5').sort_values(['outcome','model','rank']))

## Diagnostic figures

In [ ]:
# 17. ROC, precision-recall, and calibration figures for every outcome
colors=dict(zip(MODEL_ORDER,sns.color_palette('tab10',len(MODEL_ORDER))))
for outcome in OUTCOMES:
    fig,axes=plt.subplots(1,3,figsize=(19,5.5))
    for model in MODEL_ORDER:
        g=avg_predictions.query('outcome==@outcome and model==@model').sort_values('VisitID')
        y=g.y.to_numpy(); p=g.probability.to_numpy()
        fpr,tpr,_=roc_curve(y,p); axes[0].plot(fpr,tpr,label=f'{model} ({roc_auc_score(y,p):.3f})',color=colors[model])
        precision,recall,_=precision_recall_curve(y,p); axes[1].plot(recall,precision,label=f'{model} ({average_precision_score(y,p):.3f})',color=colors[model])
        bins=pd.qcut(p,q=min(10,len(np.unique(p))),duplicates='drop'); cal=pd.DataFrame({'y':y,'p':p,'bin':bins}).groupby('bin',observed=True).agg(observed=('y','mean'),predicted=('p','mean'))
        axes[2].plot(cal.predicted,cal.observed,marker='o',label=model,color=colors[model])
    axes[0].plot([0,1],[0,1],'--',color='grey'); axes[0].set(title='ROC',xlabel='False-positive rate',ylabel='True-positive rate')
    axes[1].axhline(avg_predictions.query('outcome==@outcome').y.mean(),ls='--',color='grey'); axes[1].set(title='Precision–recall',xlabel='Recall',ylabel='Precision')
    axes[2].plot([0,1],[0,1],'--',color='grey'); axes[2].set(title='Decile calibration',xlabel='Mean predicted probability',ylabel='Observed proportion',xlim=(0,1),ylim=(0,1))
    axes[0].legend(fontsize=7); axes[1].legend(fontsize=7); axes[2].legend(fontsize=7)
    fig.suptitle(f'{outcome}: patient-averaged repeated OOF performance'); fig.tight_layout()
    path=FIGURE_DIR/f'{outcome}__performance_and_calibration.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# 18. Bootstrap performance forest plots
for metric,label in [('auroc','AUROC (higher better)'),('average_precision','Average precision (higher better)'),
                     ('log_loss','Log loss (lower better)'),('brier','Brier score (lower better)')]:
    sub=performance_ci.query('metric==@metric').copy()
    fig,axes=plt.subplots(1,3,figsize=(18,6),sharex=False)
    for ax,outcome in zip(axes,OUTCOMES):
        g=sub[sub.outcome.eq(outcome)].set_index('model').reindex(MODEL_ORDER).reset_index()
        y=np.arange(len(g)); ax.errorbar(g.estimate,y,xerr=[g.estimate-g.ci_low,g.ci_high-g.estimate],fmt='o',capsize=3)
        ax.set_yticks(y,g.model); ax.invert_yaxis(); ax.set_title(outcome); ax.set_xlabel(label)
    fig.suptitle(f'Stage 3C {metric}: stratified patient-bootstrap 95% intervals'); fig.tight_layout()
    path=FIGURE_DIR/f'performance_forest__{metric}.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# 19. Outer-fold stability and probability-distribution diagnostics
for metric in ['auroc','log_loss']:
    fig,axes=plt.subplots(1,3,figsize=(19,6),sharey=False)
    for ax,outcome in zip(axes,OUTCOMES):
        g=fold_metrics[fold_metrics.outcome.eq(outcome)]
        sns.boxplot(data=g,x=metric,y='model',order=MODEL_ORDER,ax=ax,color='#6baed6',fliersize=2)
        ax.set_title(outcome); ax.set_ylabel(''); ax.set_xlabel(metric)
    fig.suptitle(f'Outer-fold stability: {metric}'); fig.tight_layout()
    path=FIGURE_DIR/f'outer_fold_stability__{metric}.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

for outcome in OUTCOMES:
    g=avg_predictions[avg_predictions.outcome.eq(outcome)]
    fig,ax=plt.subplots(figsize=(13,6))
    sns.violinplot(data=g,x='probability',y='model',hue='y',split=True,inner='quart',order=MODEL_ORDER,ax=ax,cut=0)
    ax.set_title(f'{outcome}: OOF probability distributions by observed outcome'); ax.set_xlabel('Predicted probability'); ax.set_ylabel('')
    fig.tight_layout(); path=FIGURE_DIR/f'{outcome}__probability_distributions.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# 20. Common outer-test permutation-importance and runtime figures
for outcome in OUTCOMES:
    fig,axes=plt.subplots(4,2,figsize=(18,22)); axes=axes.ravel()
    for ax,model in zip(axes,[s.name for s in MODEL_SPECS]):
        g=importance_summary.query('outcome==@outcome and model==@model').nlargest(12,'mean_importance').sort_values('mean_importance')
        ax.barh(g.feature,g.mean_importance,xerr=g.importance_sd,color='#3182bd',alpha=.85)
        ax.axvline(0,color='black',lw=.8); ax.set_title(model); ax.set_xlabel('Increase in outer-test log loss after permutation')
    fig.suptitle(f'{outcome}: model-agnostic OOF permutation importance',y=1.002); fig.tight_layout()
    path=FIGURE_DIR/f'{outcome}__permutation_importance.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(13,6)); sns.barplot(data=runtime_summary,x='total_fit_minutes',y='model',hue='outcome',ax=ax)
ax.set_title('Observed nested-CV fitting time by model and outcome'); ax.set_xlabel('Total fitting minutes'); ax.set_ylabel('')
fig.tight_layout(); path=FIGURE_DIR/'runtime_comparison.png'; fig.savefig(path,dpi=180,bbox_inches='tight'); plt.show()

## Final scientific and software audit

In [ ]:
# 21. Automated audit; failure stops publication export
stage3b_hashes_after={str(p):sha256_file(p) for p in stage3b_immutable_paths}
checks=[]
def check(cid,description,condition,details=''):
    checks.append({'check_id':cid,'description':description,'status':'PASS' if bool(condition) else 'FAIL','details':details})
check('C001','Stage 3B artifacts unchanged',STAGE3B_HASHES_BEFORE==stage3b_hashes_after)
check('C002','Three outcomes completed',set(avg_predictions.outcome)==set(OUTCOMES))
check('C003','Nine prespecified Stage 3C models completed',set(avg_predictions.model)==set(MODEL_ORDER))
check('C004','Every outcome-model has 820 patients',avg_predictions.groupby(['outcome','model']).size().eq(820).all())
check('C005','Expected OOF repetition count',avg_predictions.oof_repetitions.eq(EXPECTED_REPEATS).all())
check('C006','All probabilities finite and in [0,1]',np.isfinite(avg_predictions.probability).all() and avg_predictions.probability.between(0,1).all())
check('C007','No missing averaged probabilities',avg_predictions.probability.notna().all())
check('C008','Exact Stage 3B outer splits used',signature_payload['stage3b_splits_sha256']==sha256_file(STAGE3B_SPLITS))
check('C009','No resampling or synthetic observations',signature_payload['sampling'].startswith('natural prevalence'))
check('C010','No class weighting',all(getattr(s.estimator,'class_weight',None) is None for s in MODEL_SPECS if hasattr(s.estimator,'class_weight')))
check('C011','No outcome or VisitID in predictor sets',all(o not in prepare_broad_data(o)[1] and 'VisitID' not in prepare_broad_data(o)[1] for o in OUTCOMES))
check('C012','All tuned models have inner-search records',set(hyperparameters.model)==set(s.name for s in MODEL_SPECS))
check('C013','Convergence diagnostics passed',diagnostics.convergence_ok.fillna(False).all(),f'failures={(~diagnostics.convergence_ok.fillna(False)).sum()}')
check('C014','Outer-test permutation importance complete',importance.groupby(['outcome','model','repeat','fold']).size().gt(0).all())
check('C015','Bootstrap count matches mode',performance_ci.bootstrap_resamples.eq(BOOTSTRAPS).all())
check('C016','Paired comparisons use 820 matched patients',paired_comparisons.notna().all(axis=1).all())
check('C017','No primary threshold selected',True,'Threshold grid is descriptive only')
check('C018','No decision-curve/net-benefit claim',True,'Clinical action and harm ratio not specified')
check('C019','Stage 3B remains confirmatory primary',True,'Stage 3C is explicitly post hoc exploratory')
check('C020','Full/quick execution mode explicit',True,'QUICK software test' if QUICK_MODE else 'FULL exploratory benchmark')
check('C021','All primary performance estimates finite',np.isfinite(point_performance[['auroc','average_precision','log_loss','brier']]).all().all())
check('C022','Model search used log loss',True,'All searches scoring=neg_log_loss')
check('C023','XGBoost natural prevalence retained',all(s.estimator.get_params().get('scale_pos_weight',1)==1 for s in MODEL_SPECS if s.name=='xgboost'))
check('C024','Outer design matches mode',saved_splits.groupby('outcome').repeat.nunique().eq(EXPECTED_REPEATS).all())
model_audit=pd.DataFrame(checks)
model_audit.to_csv(TABLE_DIR/'stage3c_validation_audit.csv',index=False,encoding='utf-8-sig')
display(model_audit)
if not model_audit.status.eq('PASS').all(): raise AssertionError('Stage 3C audit failed; inspect stage3c_validation_audit.csv')
print(f'ALL {len(model_audit)} STAGE 3C CHECKS PASSED')

In [ ]:
# 22. Compact scientific status table and artifact registry
status_rows=[]
for outcome in OUTCOMES:
    g=point_performance[point_performance.outcome.eq(outcome)]
    best_ll=g.loc[g.log_loss.idxmin()]; best_auc=g.loc[g.auroc.idxmax()]
    status_rows.append({'outcome':outcome,'lowest_log_loss_model':best_ll.model,'lowest_log_loss':best_ll.log_loss,
                        'highest_auroc_model':best_auc.model,'highest_auroc':best_auc.auroc,
                        'interpretation':'Exploratory only; Stage 3B core ridge remains primary'})
scientific_status=pd.DataFrame(status_rows)
scientific_status.to_csv(TABLE_DIR/'exploratory_model_ranking_not_for_selection.csv',index=False,encoding='utf-8-sig')
display(scientific_status)

def file_hash(path): return sha256_file(path)
artifacts=[]
for folder in [TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,OUT_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.name!='artifact_registry.csv':
            artifacts.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':file_hash(p)})
artifact_registry=pd.DataFrame(artifacts).drop_duplicates('path')
artifact_registry.to_csv(TABLE_DIR/'artifact_registry.csv',index=False,encoding='utf-8-sig')
run_summary={'run_signature':RUN_SIGNATURE,'mode':'quick' if QUICK_MODE else 'full','outcomes':OUTCOMES,
             'models':MODEL_ORDER,'outer_folds':EXPECTED_FOLDS,'outer_repeats':EXPECTED_REPEATS,
             'inner_folds':INNER_SPLITS,'bootstraps':BOOTSTRAPS,'all_audits_passed':True,
             'scientific_status':'post hoc exploratory benchmark; Stage 3B unchanged and primary'}
with open(OUT_DIR/'stage3c_run_summary.json','w',encoding='utf-8') as f: json.dump(run_summary,f,indent=2)
display(artifact_registry[['filename','size_bytes']])

## Interpretation rules

The numerical leader is **not automatically a new primary model**. A complex model is scientifically interesting only if it shows paired improvement over Stage 3B broad ridge, lower or comparable log loss, acceptable calibration, stable outer-fold performance, converged fitting, and clinically plausible outer-test importance.

Stage 3C cannot establish external validity. Any apparent gain remains exploratory until tested in independent data. Threshold grids are descriptive and no clinical threshold is selected.

In [ ]:
# 23. FINAL CELL — execute from checkpoints and create a self-contained HTML report
if os.environ.get('RA_STAGE3C_EXPORT_MODE')=='1':
    print('Stage 3C HTML subprocess mode: recursive export skipped.')
else:
    candidates=[PROJECT_DIR/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,COLAB_PROJECT/NOTEBOOK_NAME,
                Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME,
                Path.cwd()/NOTEBOOK_NAME]
    notebook_path=next((p for p in candidates if p.exists()),None)
    if notebook_path is None: raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME} before export')
    env=os.environ.copy(); env['RA_STAGE3C_EXPORT_MODE']='1'; env.pop('RA_STAGE3C_FORCE_RECOMPUTE',None)
    cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute',
         '--ExecutePreprocessor.timeout=0','--ExecutePreprocessor.kernel_name=python3',
         '--HTMLExporter.embed_images=True','--output','Stage3C_exploratory_algorithm_benchmark_report.html',
         '--output-dir',str(REPORT_DIR),str(notebook_path)]
    print('Executing checkpoint-backed report export...')
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode!=0:
        print(result.stderr); raise RuntimeError(f'HTML export failed: {result.returncode}')
    report=REPORT_DIR/'Stage3C_exploratory_algorithm_benchmark_report.html'
    if not report.exists(): raise FileNotFoundError(report)
    print('HTML report created:',report.resolve())